# e-Commerce Customer Behaviour Clustering

This notebook demonstrates how to clean e-commerce transaction data, build RFM features, and cluster customers using K-Means.

The workflow is:
1. Load the dataset
2. Clean and validate the data
3. Create RFM features
4. Explore the data
5. Scale features
6. Test K values
7. Train final K-Means model
8. Save customer segments

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

project_root = Path.cwd().resolve().parent
data_path = project_root / 'data' / 'ecommerce.csv'
output_dir = project_root / 'outputs'
output_dir.mkdir(exist_ok=True, parents=True)

In [ ]:
if not data_path.exists():
    raise FileNotFoundError(f'Dataset not found at {data_path}. Please run the script in src/clustering.py once or add the data file.')

df = pd.read_csv(data_path)
df.head()

In [ ]:
df = df.drop_duplicates().copy()
df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce')
df['UnitPrice'] = pd.to_numeric(df['UnitPrice'], errors='coerce')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], errors='coerce')
df = df.dropna(subset=['CustomerID', 'InvoiceDate', 'Quantity', 'UnitPrice'])
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)].copy()
df['TotalAmount'] = df['Quantity'] * df['UnitPrice']
df.head()

In [ ]:
latest_date = df['InvoiceDate'].max()
rfm = (
    df.groupby('CustomerID')
      .agg(
          Recency=('InvoiceDate', lambda x: (latest_date - x.max()).days),
          Frequency=('InvoiceNo', 'nunique'),
          Monetary=('TotalAmount', 'sum')
      )
      .reset_index()
)
rfm.head()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
sns.histplot(rfm['Recency'], bins=20, kde=True, ax=axes[0, 0], color='steelblue')
axes[0, 0].set_title('Recency Distribution')

sns.histplot(rfm['Frequency'], bins=20, kde=True, ax=axes[0, 1], color='darkgreen')
axes[0, 1].set_title('Frequency Distribution')

sns.histplot(rfm['Monetary'], bins=20, kde=True, ax=axes[1, 0], color='darkorange')
axes[1, 0].set_title('Monetary Distribution')

sns.boxplot(data=rfm[['Recency', 'Frequency', 'Monetary']], orient='h', ax=axes[1, 1])
axes[1, 1].set_title('RFM BoxPlot')

plt.tight_layout()
plt.show()

In [ ]:
for feature in ['Recency', 'Frequency', 'Monetary']:
    q1 = rfm[feature].quantile(0.25)
    q3 = rfm[feature].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    rfm[feature] = rfm[feature].clip(lower=lower_bound, upper=upper_bound)

features = rfm[['Recency', 'Frequency', 'Monetary']].copy()
scaled_features = StandardScaler().fit_transform(features)

inertia = []
silhouette = []
for k in range(2, 11):
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(scaled_features)
    inertia.append(model.inertia_)
    silhouette.append(silhouette_score(scaled_features, labels))

pd.DataFrame({'K': list(range(2, 11)), 'Inertia': inertia, 'SilhouetteScore': silhouette})

In [ ]:
best_k = 4
final_model = KMeans(n_clusters=best_k, random_state=42, n_init=10)
rfm['Cluster'] = final_model.fit_predict(StandardScaler().fit_transform(features)) + 1
rfm.head()

In [ ]:
cluster_summary = rfm.groupby('Cluster')[['Recency', 'Frequency', 'Monetary']].mean().round(2)
cluster_summary

In [ ]:
output_file = output_dir / 'customer_segments.csv'
rfm[['CustomerID', 'Recency', 'Frequency', 'Monetary', 'Cluster']].to_csv(output_file, index=False)
print(f'Saved final customer segments to {output_file}')